# Matrix Calculus — companion notebook

Runnable companion for [`matrix-calculus.md`](./matrix-calculus.md).

1. Symbolically verify a handful of standard identities with SymPy.
2. Sanity-check them numerically against finite differences.
3. Visualize the gradient field of a quadratic form.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Symbolic verification of identities

We check $\partial(Ax)/\partial x = A$, $\partial(x^\top A x)/\partial x = (A+A^\top)x$, $\partial \,\text{tr}(AX)/\partial X = A^\top$, and $\partial \log|X|/\partial X = X^{-\top}$.

In [ ]:
n = 3
x = sp.Matrix(sp.symbols(f'x1:{n+1}'))
A = sp.Matrix(n, n, lambda i, j: sp.Symbol(f'a{i+1}{j+1}'))

# d(Ax)/dx should equal A
Ax = A * x
J = Ax.jacobian(x)
print('d(Ax)/dx == A :', sp.simplify(J - A) == sp.zeros(n, n))

# d(x^T A x)/dx should equal (A + A^T) x (returned as a row vector by .jacobian)
f = (x.T * A * x)[0, 0]
grad = sp.Matrix([sp.diff(f, xi) for xi in x])
expected = (A + A.T) * x
print('d(x^T A x)/dx == (A+A^T)x :', sp.simplify(grad - expected) == sp.zeros(n, 1))

In [ ]:
# Matrix-valued identities
p, q = 3, 3
X = sp.Matrix(p, q, lambda i, j: sp.Symbol(f'x{i+1}{j+1}'))
Acoef = sp.Matrix(q, p, lambda i, j: sp.Symbol(f'a{i+1}{j+1}'))

def grad_mat(f, M):
    return sp.Matrix(M.shape[0], M.shape[1], lambda i, j: sp.diff(f, M[i, j]))

# tr(A X): expected A^T  (here A is q x p so A^T is p x q, matching X)
f_tr = sp.trace(Acoef * X)
G = grad_mat(f_tr, X)
print('d tr(AX)/dX == A^T :', sp.simplify(G - Acoef.T) == sp.zeros(p, q))

# log|X|: expected X^{-T}
f_logdet = sp.log(sp.det(X))
G = grad_mat(f_logdet, X)
diff = sp.simplify(G - X.inv().T)
print('d log|X|/dX == X^{-T} :', diff == sp.zeros(p, q))

# ||X||_F^2 = tr(X^T X): expected 2X
f_fro = sp.trace(X.T * X)
G = grad_mat(f_fro, X)
print('d ||X||_F^2 / dX == 2X :', sp.simplify(G - 2*X) == sp.zeros(p, q))

## 2. Numerical sanity check via finite differences

For a random SPD $A$, compare the analytical gradient $(A+A^\top)x = 2Ax$ to a central-difference approximation.

In [ ]:
n = 5
M = rng.standard_normal((n, n))
Anum = M @ M.T + 0.1 * np.eye(n)   # SPD
xnum = rng.standard_normal(n)

def f(x): return x @ Anum @ x

grad_analytic = (Anum + Anum.T) @ xnum

eps = 1e-6
grad_fd = np.zeros(n)
for i in range(n):
    e = np.zeros(n); e[i] = eps
    grad_fd[i] = (f(xnum + e) - f(xnum - e)) / (2 * eps)

print('analytic :', grad_analytic)
print('finite-d :', grad_fd)
print('max |diff| :', np.max(np.abs(grad_analytic - grad_fd)))

## 3. Plot — gradient field of a 2D quadratic form

For $f(x) = \tfrac{1}{2} x^\top A x$, the gradient is $\tfrac{1}{2}(A+A^\top) x$ — arrows point toward steepest ascent of the bowl.

In [ ]:
A2 = np.array([[3.0, 1.0],
               [1.0, 2.0]])
S = 0.5 * (A2 + A2.T)

xs = np.linspace(-2, 2, 25)
X1, X2 = np.meshgrid(xs, xs)
G1 = S[0, 0] * X1 + S[0, 1] * X2
G2 = S[1, 0] * X1 + S[1, 1] * X2
F = 0.5 * (X1 * (A2[0, 0] * X1 + A2[0, 1] * X2)
           + X2 * (A2[1, 0] * X1 + A2[1, 1] * X2))

fig, ax = plt.subplots(figsize=(6, 6))
cs = ax.contour(X1, X2, F, levels=12, cmap='viridis')
ax.quiver(X1, X2, G1, G2, color='tab:red', alpha=0.6)
ax.set_aspect('equal')
ax.set_title('Level sets of (1/2) x^T A x with gradient field')
ax.set_xlabel('x_1'); ax.set_ylabel('x_2')
plt.colorbar(cs, ax=ax, shrink=0.8)
plt.show()